# 第6回: Memory Management②

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cosmac-dev/ai-agent-seminar/blob/main/session06/session06_memory.ipynb)

エージェントは複雑なタスクや一貫した体験のために過去のやり取りを覚える必要がある。メモリが無いとステートレスになり、文脈維持・経験からの学習・パーソナライズができず、単純な一問一答に留まる。核心は、単一会話の即時・一時情報と、時間をかけて蓄積する膨大で永続的な知識の両方を、どう効果的に管理するか。


---
## 0. 環境準備

In [ ]:
# @markdown 実行環境フラグ: Google Colab で実行する場合は True にする
IS_COLAB = False # @param {type:"boolean"}

In [ ]:
%pip install -q langchain langchain-core langchain-openai langgraph

In [ ]:
import os
import getpass

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY を入力する: ")

print("APIキー設定完了" if os.environ.get("OPENAI_API_KEY") else "未設定")

---
# 1. 長期記憶の実装


- 長期記憶は記憶すべき内容の抽出やバリデーションをしたうえで明示的な保存処理が必要
- 毎ターンの最初にロードし、最後に保存する

[![](https://mermaid.ink/img/pako:eNqNks9uhCAQxl9lM8dGjawKyKGX9toXaG02RKhrCrJB3O7W-O5VGuKuvZQLzDe_-fg3I9RGSGDQO-7kc8sby3V83le26nbzeHt438Xx404ZLg5aamOvIXUjeaTmSh307KYCsSor4IxR_R3glf85yIuzvHabg9yrHjxz1Yr5QhtyI3v0y7Z_uFvNQ_MzhFyYIYLGtgKYs4OMQEur-RLCuOQrcEepZQVsXgpuPyuoummuOfHu1RgdyqwZmmMIhpNYPwHYB1f9gshOSPtkhs4BK5G3ADbCBRjCJKFFnqMCI4rTAmcRXIFlKCGU4CxNU4JRhskUwbffNE1oWWQUlZSkZU5JuY9AitYZ-_LbBr4bph9o96vK?type=png)](https://mermaid.live/edit#pako:eNqNks9uhCAQxl_FzLFRg6sCcuilvfYFWhpDhLimIhsWt7s1vntdNtSuvZQLzPf9Zvg3EzRGKmBwdMKp5060VujktOOWD9Ey3h7eoyR5jHojZK2VNvYSrF-SRxrR97VeqvWBWJUVcMb0xzvAK_-roM7OisZtDnKvevAk-k4uF9qQG9mjn7b7w92i2lseWp4heGGGGFrbSWDOjioGrawW1xCmq8_B7ZVWHNiylMJ-cODDvOQcxPBqjA5p1oztPgTjQa6f8EOoQSr7ZMbBAaPEVwA2wRlYhklKy6LISpxRjEqcx3ABlmcpoQTnCCGCsxyTOYYvvydKaVXmNKsoQVVBSbWLQcnOGfty6wLfDPM3vDirhA)

In [ ]:
# @title Tool定義
import ast, operator, datetime as _dt
from langchain_core.tools import tool

# --- 複数のツールを用意する ---
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
        ast.Div: operator.truediv, ast.Pow: operator.pow, ast.USub: operator.neg}

def _safe_eval(node):
    if isinstance(node, ast.Expression): return _safe_eval(node.body)
    if isinstance(node, ast.Constant): return node.value
    if isinstance(node, ast.BinOp): return _OPS[type(node.op)](_safe_eval(node.left), _safe_eval(node.right))
    if isinstance(node, ast.UnaryOp): return _OPS[type(node.op)](_safe_eval(node.operand))
    raise ValueError("数式として解釈できない")

@tool(parse_docstring=True)
def calculator(expression: str) -> str:
    """数式を計算して結果を返す。四則演算・べき乗(**)・括弧に対応。

    Args:
        expression: 計算したい数式の文字列。例 '2 * (3 + 4)'。
    """
    try:
        return str(_safe_eval(ast.parse(expression, mode="eval")))
    except Exception as e:
        return f"計算エラー: {e}"

@tool
def current_datetime() -> str:
    """現在のローカル日時をISO形式で返す。"""
    return _dt.datetime.now().isoformat(timespec="seconds")

tools = [calculator, current_datetime]
print("用意したツール:", [t.name for t in tools])

In [ ]:
# @title 状態
from typing import Annotated, TypedDict
from langgraph.graph.message import add_messages

# TypedDict は、決まったキーと値の型を持つ辞書を表す型
class AgentState(TypedDict):
    # 会話履歴
    # Annotated は、基本の型に追加情報を添えた型
    # add_messages は、既存のメッセージ列と新しいメッセージ列を結合する関数
    messages: Annotated[list, add_messages]

    # 長期記憶から読み込んだ情報
    memories: list[str]

    # 今回の会話から抽出された記憶候補
    memory_candidates: list[str]

    # 保存してよいと判定された記憶
    approved_memories: list[str]

In [ ]:
# @title load_memory
from dataclasses import dataclass
from uuid import uuid4

from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langgraph.runtime import Runtime
from langchain_openai import ChatOpenAI

@dataclass
class Context:
    user_id: str

def load_memory(state: AgentState, runtime: Runtime[Context]) -> AgentState:
    """
    長期記憶をStoreから読む。
    現在のユーザー入力に意味的に関連する記憶だけをセマンティック検索で取得する。
    """
    user_id = runtime.context.user_id # 実行時コンテキストはグラフの実行時に渡される
    namespace = ("memories", user_id) # 保存先の名前空間 memories/{user_id} を想定

    # 直近のユーザー発話を検索クエリにする
    query = ""
    for msg in reversed(state["messages"]):
        if isinstance(msg, HumanMessage) and msg.content:
            query = str(msg.content)
            break

    # プロンプトに載せる関連記憶の最大件数
    TOP_K = 5

    if query:
        try:
            # Storeに埋め込みindexがあれば関連度順に取得される
            items = runtime.store.search(namespace, query=query, limit=TOP_K)
        except Exception:
            # index未設定などで検索できない場合は全件取得にフォールバック
            items = runtime.store.search(namespace)
    else:
        # クエリが無いターンでは検索せず全件（もしくは無し）を返す
        items = runtime.store.search(namespace)

    # "memories"フィールドだけ差分更新（上書き）
    return {
        "memories": [item.value["text"] for item in items],
    }

In [ ]:
# @title call_model
model = ChatOpenAI(
    model="gpt-5.4-mini",
    temperature=0,
    seed=42,
)

model_with_tools = model.bind_tools(tools)

def call_model(state: AgentState) -> AgentState:
    """
    通常のReAct用LLM node。
    ここでは長期記憶をプロンプトに差し込む。
    """

    memory_text = "\n".join(f"- {m}" for m in state["memories"])

    system = SystemMessage(
        content=f"""
あなたはAIエージェントです。
必要ならツールを使ってください。

参考になる長期記憶:
{memory_text}
"""
    )

    response = model_with_tools.invoke(
        [system] + state["messages"]
    )

    # "messages"フィールドだけ差分更新（追記）
    return {
        "messages": [response],
    }

In [ ]:
# @title extract_memory
# pydantic は、Pythonの型ヒントでデータスキーマを定義し、そのスキーマに基づいて検証・変換するライブラリ
from pydantic import BaseModel, Field

# BaseModel は、型付きフィールドを持つデータモデルの基底クラス
class MemoryItem(BaseModel):
    """長期記憶として保存する価値のある、ユーザーに関する1件の事実。"""

    text: str = Field(
        description="三人称・簡潔・自己完結した日本語の事実文。例: 'ユーザーはPythonが好き'"
    )


class MemoryExtraction(BaseModel):
    """会話から抽出した長期記憶の候補一覧。"""

    candidates: list[MemoryItem] = Field(
        default_factory=list,
        description="保存候補のリスト。該当が無ければ空にする。",
    )

# with_structured_output はLangChainの共通インターフェース
# 内部では、PydanticモデルをJSON Schemaに変換し、各LLMプロバイダの構造化出力APIに合わせて渡す
# このコードではChatOpenAIなので、OpenAI APIのresponse_format=json_schemaとして指定される
# OpenAI APIではresponse_formatにjson_schemaを指定すると、モデルの返答がそのJSON Schemaに制約される
# 生成結果はLangChain側でparse/validateされ、MemoryExtractionオブジェクトとして返る
# Flow: Pydanticモデル -> JSON Schema -> response_formatに指定 -> LLMがJSONを生成 -> parse/validate -> Pydanticオブジェクト
_memory_extractor = model.with_structured_output(MemoryExtraction, method="json_schema")

_MEMORY_EXTRACT_SYSTEM = """あなたは会話から「長期記憶として保存する価値のある情報」だけを抽出する抽出器です。

抽出する: ユーザーの恒常的な好み・プロフィール・目標・制約・重要な決定など、将来の会話でも役立つ事実。
抽出しない: 一時的な依頼、計算やツールの実行結果、その場限りの話題、アシスタント自身の発言、単なる推測。

各候補は三人称・簡潔・自己完結した日本語の事実文にする（例:「ユーザーはPythonが好き」）。
該当が無ければ空のリストを返す。"""


def extract_memory(state: AgentState) -> AgentState:
    """
    LLMを使用して今回の会話から保存候補を抽出する。
    構造化出力を使い、保存に値する恒常的な事実だけを取り出す。
    """

    # 抽出対象は人間とアシスタントの発話テキストに限定する
    transcript_lines = []
    for msg in state["messages"]:
        if isinstance(msg, HumanMessage) and msg.content:
            transcript_lines.append(f"User: {msg.content}")
        elif isinstance(msg, AIMessage) and msg.content:
            transcript_lines.append(f"Assistant: {msg.content}")

    # 発話が無ければLLMを呼ばずに早期return
    if not transcript_lines:
        return {"memory_candidates": []}

    transcript = "\n".join(transcript_lines)

    try:
        result = _memory_extractor.invoke(
            [
                SystemMessage(content=_MEMORY_EXTRACT_SYSTEM),
                HumanMessage(
                    content=f"次の会話から長期記憶の保存候補を抽出してください。\n\n{transcript}"
                ),
            ]
        )
        candidates = [c.text.strip() for c in result.candidates if c.text.strip()]
    except Exception:
        # 抽出に失敗した場合は安全側に倒し、何も保存候補にしない
        candidates = []

    # "memory_candidates"フィールドだけ差分更新（上書き）
    return {
        "memory_candidates": candidates,
    }

In [ ]:
# @title validate_memory
import re

# 長期保存すべきでないセンシティブ情報（PII等）を検出する正規表現
_SENSITIVE_PATTERNS = [
    re.compile(r"\b(?:\d[ -]?){13,16}\b"),                 # クレジットカード番号
    re.compile(r"\b\d{2,4}-\d{2,4}-\d{3,4}\b"),            # 電話番号など
    re.compile(r"[\w.+-]+@[\w-]+\.[\w.-]+"),               # メールアドレス
    re.compile(r"パスワード|password|秘密の質問|暗証番号", re.I),  # 認証情報
]



def _normalize(text: str) -> str:
    """重複判定用に表記ゆれを吸収して正規化する。"""
    return re.sub(r"\s+", "", text).strip().lower()


def is_sensitive(text: str) -> bool:
    """保存すべきでないセンシティブ情報（PII・認証情報）を含むか判定する。"""
    return any(p.search(text) for p in _SENSITIVE_PATTERNS)



def validate_memory(state: AgentState) -> AgentState:
    """
    保存候補を検証する。
    ポリシー（センシティブ情報）・重複をチェックし、
    保存してよいものだけを approved_memories に残す。
    """

    approved = []

    # 既存記憶を正規化して重複排除のキー集合にする
    seen = {_normalize(m) for m in state["memories"]}

    for candidate in state["memory_candidates"]:
        key = _normalize(candidate)

        # 空・既存と重複・センシティブ・一時的なものは保存しない
        if not key or key in seen:
            continue
        if is_sensitive(candidate):
            continue

        approved.append(candidate)
        seen.add(key)  # 同一バッチ内での重複も防ぐ

    # "approved_memories"フィールドだけ差分更新（上書き）
    return {
        "approved_memories": approved,
    }

In [ ]:
# @title write_memory

def write_memory(state: AgentState, runtime: Runtime[Context]) -> AgentState:
    """
    検証済みの記憶だけをStoreへ保存する。
    """

    user_id = runtime.context.user_id
    namespace = ("memories", user_id)

    for memory in state["approved_memories"]:
        runtime.store.put(
            namespace,
            str(uuid4()), # 保存先のID
            {
                "text": memory,
                "source": "conversation",
            },
        )

    # 状態は何も更新しない
    return {}


In [ ]:
# @title グラフ定義
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langchain_openai import OpenAIEmbeddings
from langgraph.store.memory import InMemoryStore
from langgraph.prebuilt import ToolNode
from langgraph.prebuilt import tools_condition


ltm_store = InMemoryStore(
    index={
        "embed": OpenAIEmbeddings(model="text-embedding-3-small"), # @param ["text-embedding-3-small", "text-embedding-3-large", "text-embedding-ada-002"]
        "dims": 1536,
        "fields": ["text"],
    }
)

stm_checkpointer = InMemorySaver()

builder = StateGraph(AgentState)

builder.add_node(load_memory)
builder.add_node(call_model)
builder.add_node("tools", ToolNode(tools))
builder.add_node(extract_memory)
builder.add_node(validate_memory)
builder.add_node(write_memory)

builder.add_edge(START, "load_memory")
builder.add_edge("load_memory", "call_model")

builder.add_conditional_edges(
    "call_model",
    tools_condition,
    {
        "tools": "tools",
        END: "extract_memory", # ENDの判定が出たらextract_memoryにルーティング
    },
)

builder.add_edge("tools", "call_model")

builder.add_edge("extract_memory", "validate_memory")
builder.add_edge("validate_memory", "write_memory")
builder.add_edge("write_memory", END)

# Store（長期記憶）とCheckpointer（短期記憶）を渡してグラフをコンパイル
graph_with_long_term_memory = builder.compile(store=ltm_store, checkpointer=stm_checkpointer)

In [ ]:
# @title グラフ実行

# @markdown スレッドIDを指定
config = {
        "configurable": {
            "thread_id": "long-term-memory-demo-001", # @param {type:"string"}
        }
    }

# @markdown ユーザーIDを指定
context = Context(user_id="user-001") # @param {type:"string"}

for chunk in graph_with_long_term_memory.stream(
    {
        "messages": [
            HumanMessage(
                content="覚えて: 私はPythonが好きです" # @param {type:"string"}
            )
        ]
    },
    context=context,
    config = config
):
    print(chunk)
    print("-" * 80)

---
# 2. 記憶可能なAIエージェント: memonic-agent

- `graph_with_long_term_memory`をパッケージ化。
- ライブラリとして利用する他、LangGraphサーバー(APIサーバー)としても使用可能

In [ ]:
if IS_COLAB:
    !git clone https://github.com/cosmac-dev/ai-agent-seminar.git
    %cd ai-agent-seminar/session06
    # セッションの再起動が要求されたら再起動後にOPENAI_API_KEYを再設定する

%pip install -e ".[server]"
!echo "OPENAI_API_KEY=$OPENAI_API_KEY" > .env

## 2.1 ライブラリとして使用

In [ ]:
from mnemonic_agent import Agent

agent = Agent()  # 既定で ChatOpenAI と埋め込み付き InMemoryStore を使用

agent.run("覚えて: 私はPythonが好きです", user_id="user-001", session_id="user-001:s1")
print(agent.run("私について覚えていることは？", user_id="user-001", session_id="user-001:s1"))
print(agent.recall("user-001"))  # 保存済みの長期記憶を一覧

## 2.2 LangGraphサーバーで動かす

In [ ]:
if IS_COLAB:
    %cd /content/ai-agent-seminar/session06
    !langgraph dev --tunnel
else:
    !langgraph dev